# 6.5 FA 支持 MXFP8 量化方案

## 1. 为什么量化 KV Cache

FA 推理（尤其 Decode）是带宽受限场景，收益账单：

- **显存**：KV Cache 数据从 BF16 变 E4M3，占用近乎减半（外加 1/32 的 scale）→ 同样显存放更长的序列、更大的 batch；
- **搬运**：MTE2 搬运量近乎减半 → memory-bound 场景下 Kernel 时间直接下降；
- **计算**：若硬件 Cube 支持 FP8 Mmad，吞吐再翻倍（累加仍 FP32，见 6.2 节黄金法则）。

Q 通常保持 BF16/FP16（每个 token 只用一次，量化收益小）；量化的主体是 **K、V（KV Cache）**。

## 2. MXFP8 存储布局

![MXFP8 布局与数据通路](images/mxfp8_layout.svg)

- **数据**：K/V 以 E4M3 存储，shape 不变（如 S2 × D）；
- **scale**：沿 D 轴（head 维）每 **32 个元素**一组，每组一个 **E8M0**（2 的幂），shape (S2, D/32)——数据量的 1/32；
- 官方算子 FusedInferAttentionScore 即采用该形态（E4M3 数据 + E8M0 scale、gs=32，配合压缩布局），与本课程蓝本一致。

## 3. Kernel 改造：相对第 5 章的 diff

第 5 章的骨架（四拍循环、乒乓、preload、mask、尾块、负载均衡）全部保留，改动集中在三处：

**① Tiling 层**
- 新增 scale tensor 的 shape / 对齐 / dtype 检查（E8M0、D 维按 32 对齐）；
- TilingKey 增加量化分支：量化与非量化走**不同模板**，编译期隔离（第 3 章「治 ICache Miss」原则的直接应用）；
- workspace 与 buffer 规划：UB 里要给 scale 留位置（很小，S2 × D/32）。

**② 搬入（MTE2）**
- K/V 按 MXFP8 搬入：数据体量减半，scale 随路一起搬（可合并在同一块或单独队列）；
- 搬运量减半是本方案在 memory-bound 下的**主要性能来源**。

**③ 计算前后的 scale 处理**——本节的核心，也是最容易写错的地方：

设 K = s_k ∘ K8（∘ 表示逐组缩放），则 mm1 的数学为：

```
score = Q · Kᵀ = Q · (s_k ∘ K8)ᵀ
```

- **K 的 scale 沿 reduce 轴（D）分组**：不同 k 位置、不同组的 scale 不同，无法在 mm 输出上「事后统一乘」。工程上两条路：
  - **随路反量化**：Vector 上先做 `s_k × K8`（E8M0 是 2 的幂 → 指数加/移位，几乎免费）恢复 BF16 再进 Mmad，mm 完全复用非量化模板；
  - **FP8 直吃**：若 Cube 支持 FP8 Mmad，按 D 分组做 mm（K 切成 D/32 段分别 Mmad，各段结果乘上本组 scale 后累加），换计算吞吐但要改 mm 编排。
- **scale 的生效位置有讲究**：K 的 scale 必须在 **softmax 之前**生效——softmax 是非线性的，各 k 位置 scale 不同，事后统一补乘 ≠ 先乘再 softmax。V 的 scale 作用在 mm2 输出（O = P · V 的 reduce 轴是 S2），分组在 D 轴上时可在输出侧按 D 分组乘回。
- **精度守则不变**：softmax 仍在 FP32 上做（第 2 章 m/l 状态、第 5 章 V1 的约定原样保留），量化只改输入端。

## 4. 调测对拍（沿用第 4 章四步法）

- golden 生成必须**模拟同一条量化链路**（E4M3 量化-反量化 + E8M0 scale），而不是拿纯 BF16 参考当 golden——否则误差里混入了「量化本身引入的、方案预期的」误差，无法定位 kernel bug；
- 建议先用 CANNSim 仿真，dump 出反量化后的 K/V 与 FP32 softmax 结果分步对拍；
- 精度预算：量化引入的相对误差通常 ~1e-2 量级（SQNR 40dB 级），比对阈值按量化方案设定，不要沿用非量化的 2e-3。

**硬件能力清单**（以课程目标硬件为准）：Cube 支持 FP8 输入、FP32 累加；E8M0 反量化可用 Vector 指令高效完成；MTE 支持 8-bit 数据搬运。具体指令与规格以所用 CANN 版本的官方文档为准。

## 小测验

1. 量化 KV Cache 的三重收益是什么？为什么 Q 通常不量化？
2. MXFP8 中 scale 的 shape 是什么？为什么选 E8M0 而不是普通 FP8 或 FP16 做 scale？
3. K 的 scale 沿 reduce 轴（D）分组，为什么不能在 mm1 输出上「事后统一乘」？给出两条工程路线及其取舍。
4. K 的 scale 为什么必须在 softmax 之前生效？V 的 scale 为什么可以在 mm2 输出侧乘回？
5. 对拍量化 FA 时，golden 应该怎么生成？为什么不能用纯 BF16 参考当 golden？